项目介绍? 

我这个项目是一个 LLM 训练加速系统, 环境是单节点四卡H100, 参考了deepseek-moe的MLA attention + MoE FFN结构, 但是我有做两个比较大的改动, 一个是attention kernel的重写, 另一个是moe的训练路径重构. 

第一块是 attention-kernel, 我参考了flash-attention2的设计, 把attention改成分块计算和online softmax的路径, 同时把kv在线重建融合到同一个kernel里, 这样可以避免把完整注意力分数矩阵写回显存的开销, 也减小了HBM访问的带宽压力

第二块是 MoE, 在传统moe, 或者dsmoe的实现中, 整个过程在dispatch和combine阶段都需要一次alltoall通信, 我参考了flashmoe的设计, 用一个持久kernel来做: 在kernel生命周期内, 管理线程块负责解析和调度任务, 计算块负责拉取任务并进行实际的计算, 因为并没有传统的nccl alltoall而是异步的nvsgmem通信, 所以可以在dispatch和combine阶段之间重叠通讯和计算, 减少通信的开销

最终结果上, 相较baseline, 整体训练step大概有1.2x的提升, 同时模型的loss曲线和baseline基本一致, 所以性能并没有明显牺牲

fa1: 在每个q row上维护至今见过的最大score和softmax分母累积和
- 对每个 Q block 逐块扫描 K/V。
- 每处理一个 K/V tile，就先算局部 score 并加 mask；
- 然后更新这一行目前见过的最大 score。
- 如果最大值变了，就把之前累计的 softmax 分母和输出累积缩放到新的参考尺度下。
- 接着计算当前 tile 的softmax分母增量, 把它加入分母累积
- 乘 V 后加入输出累积。
- 最后扫完整个 K/V 序列后，再统一归一化得到 O。

fa2: 维护未最终归一化的 output accumulator

- for each Q block:
    - 读取 Q block
    - 初始化该 Q block 每一行的 softmax 状态
      包括当前最大值、softmax 分母累积、output accumulator

    - for each K/V block:
        - 读取 K block, V block
        - 计算局部 score: S_ij = Q_i K_jᵀ
        - 加 mask，比如 causal mask / padding mask
        - 用当前 tile 的 score 更新 softmax 的参考尺度 / row max
        - 如果最大值更新了，就把之前累计的 softmax 分母和 output accumulator
          重新缩放到新的参考尺度下
        - 计算当前 tile 在新参考尺度下的 softmax 权重
        - 当前 tile 的 softmax 权重乘上 V block，得到 output contribution
        - 更新 softmax 分母累积和 output accumulator

    - 所有 K/V block 处理完后，对 output accumulator 做最终归一化
    - 写回最终 O block

fa1 vs. fa2:
- 重写 online softmax 的计算路径，减少 rescale、mask、bounds check 这类非矩阵乘开销
- 把 warp 分工从 FA1 的 sliced-K 改成 sliced-Q，让不同 warp 负责不同 query rows，减少 shared memory 交换和跨 warp 同步
- 在 sequence 维度增加并行度，让单个 head 也能被多个 thread block 并行处理

flashMoE?
- 它用actor model来组织GPU内部的分工: 大多数blocks是processor, 负责执行计算和tile通信, 最后一个block保留为OS block, OS block中第一个warp是scheduler, 负责把ready的任务分配给空闲的processor, subscriber负责接收远端发来的tile packet并将其解码为task descriptor. 
- 在gate做完路由后, token会先重排成更适合expert-major的形式. 然后源端processor会把tile封装成packet, 发送到目标gpu. 远端subscriber轮询signal以后, 先做内存一致性保障, 再把packet解码为task descriptor, 写入任务队列并更新ready flag; scheduler把已准备好的任务分配给空闲processor, 由processor执行ffn或combine任务, 并在需要时继续异步发出下一轮tile传输. 
- 从工程角度来看, 通讯使用的是NVSHMEM, 建立跨GPU的全局空间低秩, 数据写入使用DMA/RDMA; tile packet传输时伴随一个signal
- subscriber和scheduler之间的通讯走shared memory通知(因为是同一个block); 跨actor通知走global memory signal. Subscriber会轮询dispatch flags 和combine flags读取signal. 如果signal已经已经被设置, 则标记visited后生成task descriptor. 
- 为了保证数据能正确的保存和回收, 设计了一个五维的缓冲: 专家并行世界规模 * 通信轮次(dispatch & combine) * 暂存缓冲区数量 * 本地专家数 * 拓展专家容量 * token embedding维度. 以避免写写冲突. 
- 最终combine是按路由表和affinity score把expert输出加权累加回原token的输出位置

按单个token tile生命周期来看:
- 本地 token tile 经 gate / 路由整理好
- dispatch packet 发往目标 GPU
- 目标 GPU 的 Subscriber 看到 signal，确认 packet 到达
- Subscriber 解码 packet，生成 task descriptor
- Scheduler 把 ready task 分给某个空闲 Processor
- Processor 执行 expert FFN tile task
- 结果如果需要返回，就形成 combine packet
- 原 GPU 的 Subscriber 收到 combine signal，再解码
- Scheduler 再次派工
- Processor 做 combine / scale / output accumulation
- 这一块 tile 生命周期结束

信息交换: 
- fabric:
  - SM内部: shared memory和l1共享
  - 跨SM: l2 & device memory
  - GPU芯片和HBM: 内存控制器
- 芯片外:
  - pcie: cpu-gpu / gpu-gpu
  - nvlink: 带宽更大的gpu-gpu连接
  - nvswitch: 非阻塞交换网络, 以nvlink速度通信的中心交换层

dsv3.2exp?
- embedding - transformer layer(dense)\*3 - transformer layer(sparse)\*58 - hidden - lm head - MTP(如果需要的话)
- transformer layer(dense): rmsnorm - attention(DSA+MLA) - res add - rmsnorm - dense ffn - res add
- transformer layer(sparse): rmsnorm - attention(DSA+MLA) - res add - rmsnorm - dsmoe - res add

#### 面试

##### 小鹏.
1. 过一遍online softmax?
2. 对于smem的理解?
   - GPC
     - SM (Block)
        - Tensor Core
        - CUDA Core
        - Register (线程私有)
        - L1
          - SMEM
          - L1 Cache (CTA内共享)
   - L2 (on-chip, GPU全局共享)
   - HBM
3. 手写个多头注意力的代码?
4. 可以按功能块过一遍transformer的前向吗?

##### 百度. 
1. 说说你对MHA、MLA、GQA的理解? 你是怎么实现的? 压缩了什么? 有几个头?
2. 说说online softmax的实现?
3. fa1 vs. fa2?
4. dsv3.2 vs. dsv3? "稀疏"模型体现在哪?
5. 它的moe具体的实现? moe过程中的通讯? 
   - 门控gating打分 - 路由routing - dispatch - 专家计算 - combine拼回, 在dispatch和combine阶段各有一次alltoall通信
6. flashmoe中的NCCL通讯?
   - flashmoe中用的是nvshmem, 异步通讯, 没有直接的nccl
7. (寿司) 1240000数组的reduce
   - blockdim=256, 每个block处理512个元素, gridDim = ceil(1240000 / 512) = 2422
   - 每个thread_sum = input[i] + input[i + blockDim], 将thread_sum写入shared memory
   - 每个block树形规约: offset = blockDim / 2, shared[tid] += shared[tid + offset], offset /= 2, __syncthreads(), 最后shared[0]就是block的reduce结果
8. cpp: 左值 vs. 右值? 拷贝是用值传递还是引用传递?

##### 理想. 
1. 你提及了kv在线重建和因果/填充掩码, 你对kv cache相关的东西了解吗? 你的项目里有涉及这部分的设计吗? 你这个项目里又是将因果掩码放到kernel中的呢? 
   - kv在线重建在这里和kv cache有一点区别. 这里主要是在MLA里, kv不体现展开成标准attention中的那种full kv再写回显存, 而是根绝tile需要的部分, 从latent中临时重建出参与计算的kv tile. 这样做的目的是减少中间结果和HBM通讯压力, 和kv cache为了服务解码阶段复用历史token的思路不太一样. 
   - 我将mask部分融合进了attention score的计算结算, 而不是完整的mask矩阵, 具体来说, 每个block处理qk tile时, kernel能知道当前元素对应的qk position, 需要掩码的在进入softmax之前会直接置为-inf, 直接不参与这一行的max更新, exp, 和sum计算.
2. 你这个容量上限和溢出回退是怎么设计的, 能做到保持吞吐与稳定性?
   - 这里的容量上限是针对routing的负载不均衡问题, 我给每个expert设置了token数量的上限, 这个上限初始是 $$capacity ≈ ceil(total_tokens * top_k / num_experts * capacity_factor)$$
   - 在出现溢出情况时, 直接采用token dropping虽然实现简单, 但是会影响训练语义, 所以我的思路参考了dsv3的操作: 给每个expert加一个只参与topk routing的load bias, 根据每个step的专家负载动态调整bias, 过载的专家负向, 欠载的正向, 但是不参加最终的combine gate weight; 当expert在极端情况下超过容量时再退回普通moe的降级路径
   - 这里的trade off是溢出的厚尾和回退降级的比例, 需要根据实验的实际负载来进行设计和调整
3. moe训推不对齐的问题有了解吗?
   - 理论上, 在其它条件相同且没有出现溢出的情况下, 训练和推理的路由应该是一致的, 因为gating和topk都是确定的, 同一个token在训练和推理时理论上是会被路由到同一个expert.
   - 在我的系统中, 训推不对齐主要出现在容量溢出的情况
   - 但是因为观测到的溢出占比非常小(0.1%以下), 而且最后的结果证明尽管有小比例的不对齐, 训练的稳定性和模型的最终性能并没有受到明显的影响, 所以我认为这个trade off是可以接受的
4. 你这个加速系统最后效果如何? 你说比起dsv3.2-exp大约是1.2x的训练提速, 你是怎样的一个实验规模, 然后最终的模型性能呢? 有考虑过吗?
   - 对比的基线是一个参考dsv3.2的实线, 和我的模型的核心区别在于attention和moe部分的实现
   - 因为这个项目的重点是infra path, 所以模型性能并不是核心关注点, 但是我也会保证基本的正确性, 比如attention部分的输出和基线的基本一致, 小规模的训练下loss曲线没有异常, 溢出比例低时对训练的稳定性也没有明显影响.
5. FSDP和DDP这两种并行策略的核心区别是什么呢? 在你的这个项目中, 这种区别是怎么体现的呢?
   - 在ddp中, 每张GPU保留完整模型参数梯度优化器状态, 前向梵想都是每张卡计算自己mini batch, 反向时每张卡计算自己的梯度后通过allreduce同步梯度
   - fsdp是将参数梯度优化器状态按rank分片保存, 每张卡只常驻自己对应的分片数据, 前向计算到wapped module时通过allgather拼出临时的完整参数, 计算后释放; 反向时reduce scatter分片梯度, 每张卡只更新自己分片的参数; 这样的优势是显存下降明显, 但是代价是通信频率和通信调度更为复杂
6. 你为什么要做时间步和类别标签的嵌入? 这部分是不是和多模态里的输入对齐类似? 多模态的对齐有了解吗? 
   - 在扩散模型中, 不同时间步的降噪行为是不一样的, 类别嵌入标签是为了让模型额外知道目标类别, 这两个在我的网络中都是采用加注
   - 在自动驾驶环境下的多模态更多是不同的传感器/输入各自进行编码, 然后变换到某个公共空间对齐(真实世界对象一致), 做融合交互(不同视角中的信息如何进行互相补充), 最终输出感知, 预测, 规划结果
7. 你怎么验证的这个时间线重叠?
   - 主要是看nsight systems timeline, 通过不同的颜色和track来区分不同stream上的kernel执行和memcpy, 看它们在时间线上是否有重叠; 同时也会看一些关键的GPU metrics, 比如SM Active, 来验证计算资源的利用率是否有提升
8. 自动驾驶有了解吗? 相关的论文? 
   - 我的理解中, 大模型在智能驾驶环境下的一大应用是多模态场景理解, 比如将不同传感器信息等统一为场景表示
   - 因为这些数据类型复杂, 序列长, 特征维度高, 在训练/推理阶段都面临很大的计算挑战, 所以在这方面我的项目是可以做出贡献的
9.  flashmoe听起来是一个很强的kernel, 你有了解有什么模型实际用到它的吗?
    - 暂时没有, 这是一个比较新的prototype (25 june), 有开源实现和benchmark
10. 对于其它类似的加速库/语言有了解吗, 比如vllm?
    - vllm更多是一个推理库, 重点是优化llm模型的部署, 解决推理阶段的吞吐/显存管理问题
    - deepseed是一个训练库, 侧重于训练阶段的状态管理和显存管理, 比如zero, zero3就是接近fsdp的思路
11. pagedattention, flashattention的区别是什么? 你是如何理解这些区别的?
    - fa主要是训练或prefill阶段的attention kernel优化, 它通过分块和online softmax避免把完整矩阵写回hbm, 核心是减少attention计算过程里的io
    - pa偏推理serving, 更多是通过将kv拆成固定大小block来让vllm支持更大动态长度的输入